# Shannon Entropy & Self-Surprisal

Alignment as lossy compression of drive. Comparing:
- F18 logit-level entropy (model's uncertainty at next-token decision)
- Self-surprisal (model evaluating its own generated text)
- Reference surprisal (external model evaluating the text)

Shannon's English: ~1.0 bits/char

In [ ]:
import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')

# Load data
self_df = pd.read_csv('data/self_surprisal.csv')
entropy_df = pd.read_csv('data/shannon_entropy.csv')
corpus_df = pd.read_parquet('data/corpus_metrics.parquet')

# Constants
SHANNON_ENGLISH = 1.0  # bits/char
LAYER_ORDER = ['base', 'ego', 'superego', 'instruct']
LAYER_LABELS = {'base': 'BASE', 'ego': 'SFT', 'superego': 'DPO', 'instruct': 'RLVR'}

# Compute bits/char for self-surprisal
self_df['bits_per_char'] = self_df['self_surprisal'] / np.log(2) / 4
self_df['layer'] = pd.Categorical(
    self_df['model'].map(LAYER_LABELS),
    categories=['BASE', 'SFT', 'DPO', 'RLVR'], ordered=True)

print(f'Self-surprisal: {len(self_df)} rows, {self_df.family.nunique()} families')
print(f'Entropy: {len(entropy_df)} rows')
print(f'Corpus: {len(corpus_df)} rows')

## 1. Self-surprisal by family × layer (bits/char)

Shannon's English line at 1.0 bits/char. Base models hover around it; alignment pushes below.

In [ ]:
# Mean bits/char per family × layer
summary = (self_df.groupby(['family', 'layer'], observed=True)
           .bits_per_char.mean().reset_index())

(
    ggplot(summary, aes(x='layer', y='bits_per_char', group='family', color='family'))
    + geom_line(size=1, alpha=0.7)
    + geom_point(size=3)
    + geom_hline(yintercept=SHANNON_ENGLISH, linetype='dashed', color='red', alpha=0.7)
    + annotate('text', x=0.5, y=SHANNON_ENGLISH + 0.03, label='Shannon English ≈ 1.0',
               color='red', size=8, ha='left')
    + labs(x='Alignment stage', y='Self-surprisal (bits/char)',
           title='Alignment compresses below natural language',
           color='Family')
    + theme_minimal()
    + theme(figure_size=(10, 6))
)

## 2. Self-surprisal vs reference surprisal

The gap between self-surprisal and Pythia reference surprisal widens with alignment. Aligned models create a 'private language' — internally predictable, externally distinctive.

In [ ]:
# Merge self-surprisal with reference surprisal
human_fams = {'dreams', 'waking', 'c20_fiction', 'abstracts'}
ref_ai = corpus_df[~corpus_df.family.isin(human_fams)].copy()
ref_ai['_cat'] = ref_ai.label.str.replace(r'_\d+$', '', regex=True)

# Mean per family × layer
ref_summary = (ref_ai.groupby(['family', 'model'])
               .surprisal_pythia_1b_deduped.mean().reset_index()
               .rename(columns={'surprisal_pythia_1b_deduped': 'ref_surprisal'}))
self_summary = (self_df.groupby(['family', 'model'])
                .self_surprisal.mean().reset_index())

merged = self_summary.merge(ref_summary, on=['family', 'model'], how='inner')
merged['layer'] = pd.Categorical(
    merged['model'].map(LAYER_LABELS),
    categories=['BASE', 'SFT', 'DPO', 'RLVR'], ordered=True)

# Reshape for plotting
plot_df = pd.melt(merged, id_vars=['family', 'model', 'layer'],
                  value_vars=['self_surprisal', 'ref_surprisal'],
                  var_name='measure', value_name='surprisal_nats')
plot_df['measure'] = plot_df['measure'].map({
    'self_surprisal': 'Self (own model)',
    'ref_surprisal': 'Reference (Pythia 1B)'})

(
    ggplot(plot_df, aes(x='layer', y='surprisal_nats', color='measure', group='measure'))
    + geom_line(size=1, alpha=0.7)
    + geom_point(size=2.5)
    + facet_wrap('~family', ncol=5)
    + labs(x='Alignment stage', y='Surprisal (nats)',
           title='Self vs reference surprisal: the widening gap',
           subtitle='Alignment makes text more predictable to itself but not to external models',
           color='')
    + theme_minimal()
    + theme(figure_size=(16, 7), axis_text_x=element_text(rotation=45, ha='right'),
            legend_position='top')
)

## 3. The gap: self vs reference (aligned models create private language)

In [ ]:
merged['gap'] = merged['ref_surprisal'] - merged['self_surprisal']

(
    ggplot(merged, aes(x='layer', y='gap', group='family', color='family'))
    + geom_line(size=1, alpha=0.7)
    + geom_point(size=3)
    + geom_hline(yintercept=0, linetype='dashed', color='grey', alpha=0.5)
    + labs(x='Alignment stage',
           y='Gap: reference − self surprisal (nats)',
           title='Alignment creates private language',
           subtitle='Positive gap = text is more foreign to external observer than to its author',
           color='Family')
    + theme_minimal()
    + theme(figure_size=(10, 6))
)

## 4. Logit-level entropy across alignment stages (F18)

In [ ]:
entropy_df['layer'] = pd.Categorical(
    entropy_df['layer'].map(LAYER_LABELS),
    categories=['BASE', 'SFT', 'DPO', 'RLVR'], ordered=True)

ent_summary = (entropy_df.groupby(['family', 'layer'], observed=True)
               .entropy.mean().reset_index())

(
    ggplot(ent_summary, aes(x='layer', y='entropy', group='family', color='family'))
    + geom_line(size=1, alpha=0.7)
    + geom_point(size=3)
    + labs(x='Alignment stage', y='Entropy H(p) (nats)',
           title='Logit-level entropy: alignment reduces channel capacity',
           subtitle='Entropy of next-token distribution at last position',
           color='Family')
    + theme_minimal()
    + theme(figure_size=(10, 6))
)

## 5. Distribution of self-surprisal: base vs aligned

In [ ]:
# Just base vs most-aligned
ba_df = self_df[self_df.model.isin(['base', 'superego', 'instruct'])].copy()
ba_df['stage'] = ba_df['model'].apply(lambda m: 'base' if m == 'base' else 'aligned')

(
    ggplot(ba_df, aes(x='bits_per_char', fill='stage'))
    + geom_density(alpha=0.5)
    + geom_vline(xintercept=SHANNON_ENGLISH, linetype='dashed', color='red', alpha=0.7)
    + annotate('text', x=SHANNON_ENGLISH + 0.05, y=2.5, label='Shannon',
               color='red', size=8, ha='left')
    + facet_wrap('~family', ncol=5, scales='free_y')
    + labs(x='Self-surprisal (bits/char)', y='Density',
           title='Base vs aligned: distribution of self-surprisal',
           fill='Stage')
    + theme_minimal()
    + theme(figure_size=(16, 7))
    + scale_fill_manual(values=['#4e79a7', '#e15759'])
)

## 6. Amber anomaly: safety model surprises itself

In [ ]:
amber = self_df[self_df.family == 'amber'].copy()

(
    ggplot(amber, aes(x='layer', y='bits_per_char'))
    + geom_boxplot(aes(fill='layer'), alpha=0.7, show_legend=False)
    + geom_hline(yintercept=SHANNON_ENGLISH, linetype='dashed', color='red', alpha=0.7)
    + labs(x='Alignment stage', y='Self-surprisal (bits/char)',
           title='Amber: safety model is MORE surprised by its own output',
           subtitle='AmberChat (SFT, no safety) = 0.69 bits/char; AmberSafe (DPO) = 0.98')
    + theme_minimal()
    + theme(figure_size=(8, 5))
    + scale_fill_manual(values=['#4e79a7', '#f28e2b', '#e15759'])
)

## 7. Entropy reduction by content category

In [ ]:
# Compute per-prompt deltas
deltas = []
for fam in self_df.family.unique():
    base = self_df[(self_df.family == fam) & (self_df.model == 'base')].set_index('label')
    for al in ['instruct', 'superego']:
        aligned = self_df[(self_df.family == fam) & (self_df.model == al)].set_index('label')
        if not aligned.empty: break
    if aligned.empty: continue
    for label in base.index.intersection(aligned.index):
        cat = label.rsplit('_', 1)
        cat = cat[0] if len(cat) == 2 and cat[1].isdigit() else label
        deltas.append({
            'family': fam, 'category': cat,
            'delta_bits': (aligned.loc[label, 'self_surprisal'] - 
                          base.loc[label, 'self_surprisal']) / np.log(2) / 4,
        })
delta_df = pd.DataFrame(deltas)

cat_order = (delta_df.groupby('category').delta_bits.mean()
             .sort_values().index.tolist())
delta_df['category'] = pd.Categorical(delta_df['category'], 
                                       categories=cat_order, ordered=True)

(
    ggplot(delta_df, aes(x='category', y='delta_bits'))
    + geom_boxplot(fill='#4e79a7', alpha=0.6)
    + geom_hline(yintercept=0, linetype='dashed', color='grey')
    + coord_flip()
    + labs(x='', y='Δ self-surprisal (bits/char, aligned − base)',
           title='Self-surprisal reduction by content category',
           subtitle='Negative = alignment made output more predictable to itself')
    + theme_minimal()
    + theme(figure_size=(10, 6))
)

## 8. Summary: three levels of compression

In [ ]:
# Combine logit entropy, self-surprisal, and reference surprisal into one comparison
rows = []
for fam in sorted(self_df.family.unique()):
    for stage in ['base', 'aligned']:
        if stage == 'base':
            model = 'base'
        else:
            for al in ['instruct', 'superego']:
                if not self_df[(self_df.family == fam) & (self_df.model == al)].empty:
                    model = al
                    break
        
        # Self-surprisal
        ss = self_df[(self_df.family == fam) & (self_df.model == model)]
        if ss.empty: continue
        
        # Reference surprisal  
        rs = ref_ai[(ref_ai.family == fam) & (ref_ai.model == model)]
        
        # Logit entropy
        le = entropy_df[(entropy_df.family == fam) & 
                        (entropy_df.layer == LAYER_LABELS.get(model, model))]
        
        row = {'family': fam, 'stage': stage}
        row['self_surprisal'] = ss.bits_per_char.mean()
        if not rs.empty and 'surprisal_pythia_1b_deduped' in rs.columns:
            row['ref_surprisal'] = rs.surprisal_pythia_1b_deduped.mean() / np.log(2) / 4
        if not le.empty:
            row['logit_entropy'] = le.entropy.mean() / np.log(2)  # bits, not bits/char
        rows.append(row)

comp = pd.DataFrame(rows)
comp_melt = pd.melt(comp, id_vars=['family', 'stage'],
                    value_vars=['self_surprisal', 'ref_surprisal'],
                    var_name='measure', value_name='bits_per_char')
comp_melt['measure'] = comp_melt['measure'].map({
    'self_surprisal': 'Self-surprisal',
    'ref_surprisal': 'Pythia ref surprisal'})

(
    ggplot(comp_melt.dropna(), aes(x='stage', y='bits_per_char', fill='measure'))
    + geom_col(position='dodge', alpha=0.8)
    + geom_hline(yintercept=SHANNON_ENGLISH, linetype='dashed', color='red', alpha=0.7)
    + facet_wrap('~family', ncol=5)
    + labs(x='', y='bits/char',
           title='Self vs reference surprisal: base vs aligned',
           subtitle='Red line = Shannon English (1.0 bits/char)',
           fill='')
    + theme_minimal()
    + theme(figure_size=(16, 7), legend_position='top',
            axis_text_x=element_text(rotation=45, ha='right'))
    + scale_fill_manual(values=['#4e79a7', '#59a14f'])
)